# 03. Schranken und Symmetrien

In [Notebook 02](https://ktheu.github.io/bwinfN/notebooks/index.html?path=learn_bwinf1/02_backtracking.ipynb) hast du alle Lösungen gesucht oder die erste. Viele Aufgaben
der 1. Runde fragen aber nach der **besten** Lösung: die größte Punktsumme, die kürzeste Tour, die wenigsten Teile. Dann
muss man im Prinzip alle Lösungen ansehen. Zwei Ideen machen das viel schneller:

- **Schranken (Branch and Bound):** Man merkt sich die beste bisher gefundene Lösung. Kann ein angefangener Versuch selbst
  im günstigsten Fall nicht mehr besser werden, bricht man ihn ab.
- **Symmetrien:** Oft sind zwei Lösungen „eigentlich gleich“, zum Beispiel gespiegelt oder gedreht. Dann sucht man nur
  eine davon oder bringt jede in eine feste **Normalform**, damit man sie nicht doppelt zählt.

In diesem Notebook lernst du:

- die beste Lösung mit `beste = [wert, loesung]` zu merken,
- eine **Schranke** zu finden und zu begründen, dass sie keine Lösung verliert,
- mit `sorted(..., key=lambda ...)` die Reihenfolge zu wählen, in der probiert wird,
- eine Symmetrie in der Suche zu **brechen** (nur die Hälfte durchsuchen),
- Figuren aus Quadraten zu **verschieben, drehen und spiegeln** und die kleinste Lage als Normalform zu nehmen.

In der Wettbewerbs-Ecke bepflanzt du ein Blumenbeet mit möglichst vielen Punkten und zählst Rominos.

## 1. Die beste Lösung suchen

Für eine Wanderung darf dein Rucksack höchstens **12 kg** wiegen. Jeder Gegenstand hat ein Gewicht und einen Nutzen
(Punkte). Welche Auswahl hat den größten Nutzen?

| Gegenstand | Zelt | Schlafsack | Kocher | Topf | Wasser | Brot | Kamera | Buch | Jacke | Karte | Lampe | Seil | Apfel | Kekse |
|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|
| Gewicht (kg) | 5 | 3 | 2 | 1 | 3 | 1 | 2 | 1 | 2 | 1 | 1 | 2 | 1 | 1 |
| Nutzen | 10 | 8 | 5 | 2 | 9 | 4 | 6 | 1 | 7 | 3 | 4 | 3 | 2 | 3 |

Das ist das **Rucksackproblem**. Wie bei den Teilsummen entscheidet jeder Aufruf über einen Gegenstand: einpacken (wenn er
noch passt) oder nicht. Neu ist: Wir suchen nicht alle Auswahlen, sondern die **beste**. Dafür merken wir sie uns in einer
Liste `beste = [nutzen, auswahl]`. Jede Auswahl, die in den Rucksack passt, ist eine erlaubte Lösung. Darum vergleichen wir
in **jedem** Aufruf: Ist der Nutzen größer als der beste bisher, wird er mit einer Kopie der Auswahl gespeichert.

#### Aufgabe 1.1

Schreibe `packe(i, gewicht, nutzen, gewaehlt)`. Sie vergleicht zuerst mit `beste`. Sind noch Gegenstände übrig, probiert sie
Gegenstand `i` einzupacken (nur wenn das Gewicht dann höchstens `KAPAZITAET` ist) und danach, ihn wegzulassen.

**Erwartete Ausgabe:**
```
38 Punkte: ['Schlafsack', 'Wasser', 'Brot', 'Kamera', 'Jacke', 'Lampe']
16600 Aufrufe
```

In [ ]:
dinge = [('Zelt', 5, 10), ('Schlafsack', 3, 8), ('Kocher', 2, 5), ('Topf', 1, 2), ('Wasser', 3, 9), ('Brot', 1, 4),
         ('Kamera', 2, 6), ('Buch', 1, 1), ('Jacke', 2, 7), ('Karte', 1, 3), ('Lampe', 1, 4), ('Seil', 2, 3),
         ('Apfel', 1, 2), ('Kekse', 1, 3)]
KAPAZITAET = 12
beste = [0, []]
aufrufe = [0]

def packe(i, gewicht, nutzen, gewaehlt):
    aufrufe[0] = aufrufe[0] + 1
    # dein Code

packe(0, 0, 0, [])
print(beste[0], 'Punkte:', beste[1])
print(aufrufe[0], 'Aufrufe')

<details>
<summary>Tipp anzeigen</summary>

`name, g, n = dinge[i]` packt den Gegenstand aus. Einpacken heißt: `gewaehlt.append(name)`, dann
`packe(i + 1, gewicht + g, nutzen + n, gewaehlt)`, dann `gewaehlt.pop()`. Weglassen: `packe(i + 1, gewicht, nutzen, gewaehlt)`.

</details>

<details>
<summary>Lösung anzeigen (erst selbst probieren!)</summary>

```python
dinge = [('Zelt', 5, 10), ('Schlafsack', 3, 8), ('Kocher', 2, 5), ('Topf', 1, 2), ('Wasser', 3, 9), ('Brot', 1, 4),
         ('Kamera', 2, 6), ('Buch', 1, 1), ('Jacke', 2, 7), ('Karte', 1, 3), ('Lampe', 1, 4), ('Seil', 2, 3),
         ('Apfel', 1, 2), ('Kekse', 1, 3)]
KAPAZITAET = 12
beste = [0, []]
aufrufe = [0]

def packe(i, gewicht, nutzen, gewaehlt):
    aufrufe[0] = aufrufe[0] + 1
    if nutzen > beste[0]:                          # besser als alles bisher?
        beste[0] = nutzen
        beste[1] = list(gewaehlt)                  # Kopie!
    if i == len(dinge):
        return
    name, g, n = dinge[i]
    if gewicht + g <= KAPAZITAET:                  # einpacken, wenn es passt
        gewaehlt.append(name)
        packe(i + 1, gewicht + g, nutzen + n, gewaehlt)
        gewaehlt.pop()
    packe(i + 1, gewicht, nutzen, gewaehlt)        # weglassen

packe(0, 0, 0, [])
print(beste[0], 'Punkte:', beste[1])
print(aufrufe[0], 'Aufrufe')
```

Ohne Gewichtsgrenze wären es 1 + 2 + 4 + … + 2¹⁴ = 32 767 Aufrufe: die 2¹⁴ = 16 384 fertigen Auswahlen und alle halb
fertigen. Weil zu schwere Auswahlen gar nicht erst weiterverfolgt werden, sind es nur 16 600. Bei 50 Gegenständen wären es aber schon Billionen.

</details>

## 2. Schranken: Kann das noch besser werden?

Angenommen, die beste Auswahl bisher hat 38 Punkte. Jetzt sind wir bei Gegenstand 10 und haben 25 Punkte im Rucksack. Die
Gegenstände 10 bis 13 haben zusammen nur 12 Punkte Nutzen. Selbst wenn wir sie **alle** einpacken könnten, kämen wir
höchstens auf 37 Punkte. Dieser Versuch kann die beste Auswahl also nicht mehr schlagen, und wir brechen ab.

Das ist **Branch and Bound** (verzweigen und beschränken). Die **Schranke** ist eine Zahl, die angibt, wie gut ein
angefangener Versuch **höchstens** noch werden kann. Ist sie nicht größer als das Beste bisher, wird abgebrochen.

Damit das richtig ist, muss die Schranke **optimistisch** sein: Sie darf zu groß sein, aber nie zu klein. Hier ist sie
`nutzen + rest[i]`, wobei `rest[i]` der Nutzen aller Gegenstände ab `i` ist. Das Gewicht wird dabei einfach ignoriert. Das
macht die Schranke größer als nötig, aber nie zu klein.

#### Aufgabe 2.1

Schreibe `packe2` wie `packe`, aber mit dem Abbruch `nutzen + rest[i] <= beste[0]`, direkt nach dem Test, ob noch
Gegenstände übrig sind. Die Liste `rest` wird schon berechnet.

**Erwartete Ausgabe:**
```
38 Punkte: ['Schlafsack', 'Wasser', 'Brot', 'Kamera', 'Jacke', 'Lampe']
2514 Aufrufe
```

In [ ]:
rest = [0] * (len(dinge) + 1)
for i in range(len(dinge) - 1, -1, -1):
    rest[i] = rest[i + 1] + dinge[i][2]

def packe2(i, gewicht, nutzen, gewaehlt):
    aufrufe[0] = aufrufe[0] + 1
    # dein Code

beste = [0, []]
aufrufe[0] = 0
packe2(0, 0, 0, [])
print(beste[0], 'Punkte:', beste[1])
print(aufrufe[0], 'Aufrufe')

<details>
<summary>Lösung anzeigen (erst selbst probieren!)</summary>

```python
rest = [0] * (len(dinge) + 1)
for i in range(len(dinge) - 1, -1, -1):
    rest[i] = rest[i + 1] + dinge[i][2]

def packe2(i, gewicht, nutzen, gewaehlt):
    aufrufe[0] = aufrufe[0] + 1
    if nutzen > beste[0]:
        beste[0] = nutzen
        beste[1] = list(gewaehlt)
    if i == len(dinge):
        return
    if nutzen + rest[i] <= beste[0]:               # Schranke: selbst alles Übrige reicht nicht
        return
    name, g, n = dinge[i]
    if gewicht + g <= KAPAZITAET:
        gewaehlt.append(name)
        packe2(i + 1, gewicht + g, nutzen + n, gewaehlt)
        gewaehlt.pop()
    packe2(i + 1, gewicht, nutzen, gewaehlt)

beste = [0, []]
aufrufe[0] = 0
packe2(0, 0, 0, [])
print(beste[0], 'Punkte:', beste[1])
print(aufrufe[0], 'Aufrufe')
```

Dasselbe Ergebnis mit einem Siebtel der Aufrufe. Die Schranke wirkt umso besser, je früher eine **gute** Lösung gefunden
wird, denn erst dann ist `beste[0]` groß genug zum Abschneiden.

</details>

### Die Reihenfolge wählen: `sorted` mit `key=lambda`

Findet die Suche früh eine gute Lösung, schneidet die Schranke mehr ab. Darum lohnt es sich, die **wertvollsten**
Gegenstände zuerst zu probieren. Dazu sortiert man die Liste:

```python
dinge = sorted(dinge, key=lambda d: d[2], reverse=True)
```

- `key=` sagt `sorted`, **wonach** sortiert wird. Ohne `key` würden die Tupel nach ihrem ersten Element sortiert, also nach
  dem Namen.
- `lambda d: d[2]` ist eine kleine Funktion ohne Namen: Sie bekommt einen Gegenstand `d` und gibt `d[2]` zurück, seinen
  Nutzen. `sorted` ruft sie für jeden Gegenstand auf und sortiert nach diesen Zahlen.
- `reverse=True` sortiert absteigend, der größte Nutzen zuerst.

Ohne `lambda` geht es auch: Man baut Tupel, deren **erstes** Element der Schlüssel ist, und sortiert diese:
`paare = []`, dann für jeden Gegenstand `paare.append((d[2], d))`, dann `paare.sort(reverse=True)`. Das ist länger, darum
benutzen wir in diesem Lehrgang `key=lambda`.

#### Aufgabe 2.2

Sortiere `dinge` absteigend nach dem Nutzen, berechne `rest` neu und rufe `packe2` noch einmal auf. Gib die sortierten Namen,
das Ergebnis und die Aufrufe aus.

**Erwartete Ausgabe:**
```
['Zelt', 'Wasser', 'Schlafsack', 'Jacke', 'Kamera', 'Kocher', 'Brot', 'Lampe', 'Karte', 'Seil', 'Kekse', 'Topf', 'Apfel', 'Buch']
38 Punkte: ['Wasser', 'Schlafsack', 'Jacke', 'Kamera', 'Brot', 'Lampe']
1264 Aufrufe
```

In [ ]:
# dein Code

beste = [0, []]
aufrufe[0] = 0
packe2(0, 0, 0, [])
print(beste[0], 'Punkte:', beste[1])
print(aufrufe[0], 'Aufrufe')

<details>
<summary>Tipp anzeigen</summary>

Zuerst `dinge = sorted(dinge, key=lambda d: d[2], reverse=True)`, dann die drei Zeilen für `rest` aus 2.1 noch einmal.

</details>

<details>
<summary>Lösung anzeigen (erst selbst probieren!)</summary>

```python
dinge = sorted(dinge, key=lambda d: d[2], reverse=True)
namen = []
for d in dinge:
    namen.append(d[0])
print(namen)
rest = [0] * (len(dinge) + 1)
for i in range(len(dinge) - 1, -1, -1):
    rest[i] = rest[i + 1] + dinge[i][2]

beste = [0, []]
aufrufe[0] = 0
packe2(0, 0, 0, [])
print(beste[0], 'Punkte:', beste[1])
print(aufrufe[0], 'Aufrufe')
```

Wieder 38 Punkte mit denselben Gegenständen, aber nur noch halb so viele Aufrufe wie in 2.1. Eine andere
naheliegende Reihenfolge ist „Nutzen pro Kilogramm“ (`key=lambda d: d[2] / d[1]`). Sie ergibt hier 1770 Aufrufe, ist also
etwas schlechter. Welche Reihenfolge am besten ist, hängt von den Daten ab. Man probiert es aus und zählt.

</details>

#### Frage 2.3

Für die Dokumentation:

1. Warum findet `packe2` trotz des Abbrechens sicher die beste Auswahl?
2. Warum darf man bei `nutzen + rest[i] == beste[0]` abbrechen, also `<=` statt `<` schreiben?
3. Wird die Laufzeit durch die Schranke besser als O(2ⁿ)?

✏️ *Deine Antwort (Doppelklick zum Bearbeiten):*

<details>
<summary>Lösung anzeigen (erst selbst überlegen!)</summary>

1. Abgebrochen wird nur, wenn `nutzen + rest[i] <= beste[0]` ist. Jede Fortsetzung kann höchstens die Gegenstände ab `i`
   dazunehmen, also höchstens `rest[i]` Punkte. Sie kommt darum höchstens auf `beste[0]` und ist nicht besser. Keine
   bessere Auswahl geht verloren.
2. Bei Gleichstand ist eine Fortsetzung höchstens **genauso gut** wie die beste bisher. Wir wollen nur **eine** beste
   Auswahl, also brauchen wir sie nicht. (Sollen alle besten Auswahlen gefunden werden, muss man `<` nehmen.)
3. Im schlimmsten Fall nein: Es gibt Eingaben, bei denen die Schranke fast nie greift, dann bleibt es O(2ⁿ). In der Praxis
   spart sie oft sehr viel. Darum schreibt man in die Dokumentation die obere Grenze **und** die gemessenen Zähler für die
   Beispiele.

</details>

#### Frage 2.4

Finde den Fehler. Mia wollte sich `rest` sparen und schreibt als Schranke „Nutzen jetzt plus Nutzen des nächsten
Gegenstands“:

```python
def packe_falsch(i, gewicht, nutzen, gewaehlt):
    if nutzen > beste[0]:
        beste[0] = nutzen
        beste[1] = list(gewaehlt)
    if i == len(dinge):
        return
    if nutzen + dinge[i][2] <= beste[0]:           # Schranke?
        return
    name, g, n = dinge[i]
    if gewicht + g <= KAPAZITAET:
        gewaehlt.append(name)
        packe_falsch(i + 1, gewicht + g, nutzen + n, gewaehlt)
        gewaehlt.pop()
    packe_falsch(i + 1, gewicht, nutzen, gewaehlt)
```

Mit der ursprünglichen Reihenfolge der Gegenstände findet ihr Programm nur 29 statt 38 Punkte. Warum?

✏️ *Deine Antwort (Doppelklick zum Bearbeiten):*

<details>
<summary>Lösung anzeigen (erst selbst überlegen!)</summary>

Die Schranke ist **zu klein** (pessimistisch). Nach Gegenstand `i` können noch viele weitere dazukommen, nicht nur einer.
Mias Programm bricht also auch Versuche ab, die noch besser geworden wären. Das Programm wird schnell, aber falsch, und
merkt es nicht einmal. Merke: Eine Schranke muss man **beweisen** können. Und man testet sie, indem man bei kleinen
Beispielen die Ergebnisse mit und ohne Schranke vergleicht.

</details>

## 3. Symmetrien

In Notebook 02 hatte das 6×6-Damenproblem 4 Lösungen, aber die sind alle gleich, wenn man das Brett dreht oder spiegelt.
Beim Dreieckspuzzle kam jede Lösung dreimal vor. Solche **Symmetrien** kann man auf zwei Arten nutzen:

1. **Die Symmetrie in der Suche brechen:** Man sucht nur einen Teil der Lösungen und weiß, wie man daraus die anderen
   bekommt. Das spart Zeit.
2. **Normalform:** Man bringt jede gefundene Lösung in eine feste Form, sodass „gleiche“ Lösungen genau gleich aussehen.
   Dann zählt eine Menge (`set`) jede nur einmal. Das braucht man, wenn die Aufgabe nach **verschiedenen** Lösungen fragt.

### Spiegeln beim Damenproblem

Spiegelt man ein Brett an der senkrechten Mittellinie, wird aus der Spalte `s` die Spalte `n - 1 - s`, und aus jeder Lösung
wird wieder eine Lösung. Bei geradem `n` steht die Dame der ersten Zeile entweder in der linken oder in der rechten Hälfte.
Also genügt es, die linke Hälfte zu durchsuchen und die Lösungen zu verdoppeln.

In [ ]:
aufrufe = [0]

def passt(damen, spalte):
    zeile = len(damen)
    for r in range(zeile):
        if damen[r] == spalte or abs(damen[r] - spalte) == zeile - r:
            return False
    return True

def setze(damen, n, erste_spalten, loesungen):
    aufrufe[0] = aufrufe[0] + 1
    if len(damen) == n:
        loesungen.append(list(damen))
        return
    if len(damen) == 0:
        spalten = erste_spalten        # erste Zeile: nur diese Spalten
    else:
        spalten = range(n)
    for spalte in spalten:
        if passt(damen, spalte):
            damen.append(spalte)
            setze(damen, n, erste_spalten, loesungen)
            damen.pop()

for erste_spalten in [range(8), range(4)]:
    aufrufe[0] = 0
    loesungen_damen = []
    setze([], 8, erste_spalten, loesungen_damen)
    print('erste Dame in', list(erste_spalten), ':', len(loesungen_damen), 'Lösungen,', aufrufe[0], 'Aufrufe')

#### Frage 3.1

1. Warum ist 2 · 46 = 92 sicher richtig, warum also wird keine Lösung doppelt oder gar nicht gezählt?
2. Was ist bei ungeradem `n`, zum Beispiel n = 7?

✏️ *Deine Antwort (Doppelklick zum Bearbeiten):*

<details>
<summary>Lösung anzeigen (erst selbst überlegen!)</summary>

1. Jede Lösung hat ihre erste Dame entweder links (Spalten 0 bis 3) oder rechts (4 bis 7). Die rechten Lösungen sind genau
   die Spiegelbilder der linken, und eine Lösung ist nie ihr eigenes Spiegelbild, denn die erste Dame wechselt die Hälfte.
   Darum gibt es links und rechts gleich viele.
2. Bei n = 7 gibt es eine **mittlere** Spalte 3, die beim Spiegeln bleibt. Man durchsucht die Spalten 0 bis 2 und
   verdoppelt, und zählt die Lösungen mit der ersten Dame in Spalte 3 einmal dazu. Symmetrien zu brechen ist also wirksam,
   aber man muss sorgfältig begründen, dass dabei nichts verloren geht oder doppelt gezählt wird.

</details>

### Normalform: Figuren aus Quadraten

Eine Figur aus Quadraten speichern wir als Liste von Feldern `(x, y)`: `x` zählt nach rechts, `y` nach oben. Das L aus vier
Quadraten ist zum Beispiel `[(0, 0), (0, 1), (0, 2), (1, 0)]`.

Dieselbe Figur kann an verschiedenen Stellen liegen und verschieden gedreht oder gespiegelt sein. Um zu erkennen, ob zwei
Listen dieselbe Figur beschreiben, bringen wir beide in eine **Normalform**:

1. **Verschieben:** Die Figur so verschieben, dass das kleinste `x` und das kleinste `y` beide 0 sind. Dann die Felder
   sortieren und als **Tupel** speichern. Sortieren ist nötig, weil dieselben Felder sonst in anderer Reihenfolge
   verschieden aussehen. Ein Tupel braucht man, weil Listen nicht in eine Menge dürfen.
2. **Drehen und Spiegeln:** Eine Drehung um 90° macht aus `(x, y)` das Feld `(y, -x)`, eine Spiegelung macht daraus
   `(-x, y)`. Viermal drehen, spiegeln, noch viermal drehen ergibt alle **8 Lagen** der Figur.
3. **Die kleinste Lage wählen:** Tupel kann man vergleichen wie Wörter im Wörterbuch, Stelle für Stelle. Von den 8
   verschobenen Lagen nehmen wir mit `min` die kleinste. Zwei Figuren sind genau dann gleich, wenn ihre kleinsten Lagen
   gleich sind.

#### Aufgabe 3.2

Schreibe `normalisiere(figur)`, die die Figur verschiebt, sodass das kleinste `x` und das kleinste `y` 0 sind, und die Felder
sortiert als Tupel zurückgibt.

**Erwartete Ausgabe:**
```
((0, 0), (0, 1), (1, 0))
((0, 0), (0, 1), (1, 0))
```

In [ ]:
def normalisiere(figur):
    # dein Code
    return tuple(figur)

print(normalisiere([(3, 5), (2, 5), (2, 6)]))
print(normalisiere([(0, 1), (0, 0), (1, 0)]))

<details>
<summary>Tipp anzeigen</summary>

Erst in einer Schleife das kleinste `x` und das kleinste `y` suchen, dann eine neue Liste mit `(x - kleinstes_x, y - kleinstes_y)`
bauen und `tuple(sorted(...))` zurückgeben.

</details>

<details>
<summary>Lösung anzeigen (erst selbst probieren!)</summary>

```python
def normalisiere(figur):
    kleinstes_x = figur[0][0]
    kleinstes_y = figur[0][1]
    for x, y in figur:
        if x < kleinstes_x:
            kleinstes_x = x
        if y < kleinstes_y:
            kleinstes_y = y
    verschoben = []
    for x, y in figur:
        verschoben.append((x - kleinstes_x, y - kleinstes_y))
    return tuple(sorted(verschoben))

print(normalisiere([(3, 5), (2, 5), (2, 6)]))
print(normalisiere([(0, 1), (0, 0), (1, 0)]))
```

Beide Listen beschreiben dieselbe Figur an verschiedenen Stellen, und beide ergeben dasselbe Tupel.

</details>

#### Aufgabe 3.3

Schreibe `kanonisch(figur)`, die alle 8 Lagen der Figur normalisiert und die kleinste zurückgibt. `drehe` und `spiegle` sind
fertig. Teste mit zwei Lagen des L und mit dem T.

**Erwartete Ausgabe:**
```
((0, 0), (0, 1), (0, 2), (1, 0))
True False
```

In [ ]:
def drehe(figur):
    gedreht = []
    for x, y in figur:
        gedreht.append((y, -x))
    return gedreht

def spiegle(figur):
    gespiegelt = []
    for x, y in figur:
        gespiegelt.append((-x, y))
    return gespiegelt

def kanonisch(figur):
    lagen = []
    # dein Code
    return normalisiere(figur)       # ersetze diese Zeile

l_figur = [(0, 0), (0, 1), (0, 2), (1, 0)]
l_gedreht = [(0, 0), (1, 0), (2, 0), (2, 1)]
t_figur = [(0, 0), (1, 0), (2, 0), (1, 1)]
print(kanonisch(l_figur))
print(kanonisch(l_gedreht) == kanonisch(l_figur), kanonisch(t_figur) == kanonisch(l_figur))

<details>
<summary>Tipp anzeigen</summary>

Zwei Schleifen ineinander: `for spiegelung in range(2):` und darin `for drehung in range(4):`. In der inneren Schleife
`lagen.append(normalisiere(f))` und `f = drehe(f)`, nach der inneren Schleife `f = spiegle(f)`.

</details>

<details>
<summary>Lösung anzeigen (erst selbst probieren!)</summary>

```python
def drehe(figur):
    gedreht = []
    for x, y in figur:
        gedreht.append((y, -x))
    return gedreht

def spiegle(figur):
    gespiegelt = []
    for x, y in figur:
        gespiegelt.append((-x, y))
    return gespiegelt

def kanonisch(figur):
    lagen = []
    f = list(figur)
    for spiegelung in range(2):
        for drehung in range(4):
            lagen.append(normalisiere(f))
            f = drehe(f)
        f = spiegle(f)
    return min(lagen)

l_figur = [(0, 0), (0, 1), (0, 2), (1, 0)]
l_gedreht = [(0, 0), (1, 0), (2, 0), (2, 1)]
t_figur = [(0, 0), (1, 0), (2, 0), (1, 1)]
print(kanonisch(l_figur))
print(kanonisch(l_gedreht) == kanonisch(l_figur), kanonisch(t_figur) == kanonisch(l_figur))
```

</details>

### Alle Figuren erzeugen

Alle Figuren aus `n` Quadraten bekommst du, indem du die Figuren aus `n - 1` Quadraten **wachsen** lässt: An jedes Feld wird
in jeder Richtung ein Nachbarfeld angesetzt, wenn es noch frei ist. Das Ergebnis wird normalisiert und in eine Menge gelegt.
So fallen alle Figuren weg, die nur **verschoben** sind. Die Menge enthält dann alle verschiedenen **Lagen**. Mit
`kanonisch` fallen auch die gedrehten und gespiegelten weg, übrig bleiben die verschiedenen **Figuren**.

Die Richtungen stehen in einer Liste. Für Figuren, deren Quadrate an den Seiten zusammenhängen, sind es vier.

#### Aufgabe 3.4

Schreibe `wachse(figuren, richtungen)`, die aus einer Menge von Figuren (normalisierten Tupeln) die Menge aller Figuren mit
einem Quadrat mehr macht. Gib für n = 2 bis 6 die Zahl der Lagen und der Figuren aus und zeichne die Figuren aus 5 Quadraten
mit der fertigen Funktion `zeichne`.

**Erwartete Ausgabe:**
```
2 Quadrate: 2 Lagen, 1 Figuren
3 Quadrate: 6 Lagen, 2 Figuren
4 Quadrate: 19 Lagen, 5 Figuren
5 Quadrate: 63 Lagen, 12 Figuren
6 Quadrate: 216 Lagen, 35 Figuren
#
#   #.   #.                         .#
#   #.   #.   #.   ##   #..   #..   ##   .#.   .##   ..#   .#.
#   #.   ##   ##   #.   #..   ###   #.   ###   ##.   ###   ###
#   ##   #.   ##   ##   ###   #..   #.   #..   #..   #..   .#.
```

In [ ]:
SEITEN = [(1, 0), (-1, 0), (0, 1), (0, -1)]

def zeichne(figuren):
    hoehe = 0
    for figur in figuren:
        for x, y in figur:
            if y + 1 > hoehe:
                hoehe = y + 1
    for zeile in range(hoehe - 1, -1, -1):
        text = ''
        for figur in figuren:
            breite = 0
            hoch = 0
            for x, y in figur:
                breite = max(breite, x + 1)
                hoch = max(hoch, y + 1)
            for spalte in range(breite):
                if zeile >= hoch:
                    text = text + ' '
                elif (spalte, zeile) in figur:
                    text = text + '#'
                else:
                    text = text + '.'
            text = text + '   '
        print(text.rstrip())

def wachse(figuren, richtungen):
    neu = set()
    # dein Code
    return neu

figuren = {((0, 0),)}
for n in range(2, 7):
    figuren = wachse(figuren, SEITEN)
    verschieden = set()
    for figur in figuren:
        verschieden.add(kanonisch(figur))
    print(n, 'Quadrate:', len(figuren), 'Lagen,', len(verschieden), 'Figuren')
    if n == 5:
        fuenf = sorted(verschieden)
zeichne(fuenf)

<details>
<summary>Tipp anzeigen</summary>

Drei Schleifen ineinander: über die Figuren, über ihre Felder `(x, y)` und über die Richtungen `(dx, dy)`. Ist
`feld = (x + dx, y + dy)` noch nicht in der Figur, kommt `normalisiere(list(figur) + [feld])` in die Menge `neu`.

</details>

<details>
<summary>Lösung anzeigen (erst selbst probieren!)</summary>

```python
SEITEN = [(1, 0), (-1, 0), (0, 1), (0, -1)]

def zeichne(figuren):
    hoehe = 0
    for figur in figuren:
        for x, y in figur:
            if y + 1 > hoehe:
                hoehe = y + 1
    for zeile in range(hoehe - 1, -1, -1):
        text = ''
        for figur in figuren:
            breite = 0
            hoch = 0
            for x, y in figur:
                breite = max(breite, x + 1)
                hoch = max(hoch, y + 1)
            for spalte in range(breite):
                if zeile >= hoch:
                    text = text + ' '
                elif (spalte, zeile) in figur:
                    text = text + '#'
                else:
                    text = text + '.'
            text = text + '   '
        print(text.rstrip())

def wachse(figuren, richtungen):
    neu = set()
    for figur in figuren:
        for x, y in figur:
            for dx, dy in richtungen:
                feld = (x + dx, y + dy)
                if feld not in figur:
                    neu.add(normalisiere(list(figur) + [feld]))
    return neu

figuren = {((0, 0),)}
for n in range(2, 7):
    figuren = wachse(figuren, SEITEN)
    verschieden = set()
    for figur in figuren:
        verschieden.add(kanonisch(figur))
    print(n, 'Quadrate:', len(figuren), 'Lagen,', len(verschieden), 'Figuren')
    if n == 5:
        fuenf = sorted(verschieden)
zeichne(fuenf)
```

Das sind die 12 **Pentominos** (Figuren aus 5 Quadraten), dazu 5 Tetrominos aus 4 Quadraten, wie bei dem bekannten
Computerspiel. Eine Figur hat höchstens 8 verschiedene Lagen. Es sind aber nur 63 Lagen und nicht 8 · 12 = 96, denn symmetrische
Figuren haben weniger: Das gerade I hat nur 2 Lagen, das Plus nur eine.

</details>

#### Frage 3.5

Finde den Fehler. Tom hat in `normalisiere` das Sortieren weggelassen:

```python
def normalisiere(figur):
    kleinstes_x = figur[0][0]
    kleinstes_y = figur[0][1]
    for x, y in figur:
        if x < kleinstes_x:
            kleinstes_x = x
        if y < kleinstes_y:
            kleinstes_y = y
    verschoben = []
    for x, y in figur:
        verschoben.append((x - kleinstes_x, y - kleinstes_y))
    return tuple(verschoben)
```

Jetzt meldet sein Programm bei 4 Quadraten 176 Lagen statt 19. Warum?

✏️ *Deine Antwort (Doppelklick zum Bearbeiten):*

<details>
<summary>Lösung anzeigen (erst selbst überlegen!)</summary>

Ohne Sortieren hängt das Tupel von der **Reihenfolge** ab, in der die Felder entstanden sind. Dieselbe Lage, einmal als
`((0, 0), (1, 0))` und einmal als `((1, 0), (0, 0))`, sind für Python zwei verschiedene Tupel und landen beide in der
Menge. Eine Normalform muss **eindeutig** sein: Gleiche Dinge müssen genau gleich aussehen. Ein guter Test dafür sind
bekannte Zahlen (hier 19 Lagen und 5 Figuren aus 4 Quadraten).

</details>

## Wettbewerbs-Ecke 1: Blumenbeet

*Angelehnt an Aufgabe 1 „Blumenbeet“ der 1. Runde des 38. Bundeswettbewerbs Informatik (2019/20).*

In ein rautenförmiges Hochbeet werden neun Blumen gepflanzt, in Reihen aus 1, 2, 3, 2 und 1 Blumen. Die Plätze sind so
nummeriert:

                0
            1       2
        3       4       5
            6       7
                8

Nachbarn sind die Blumen links und rechts daneben und die bis zu vier Blumen schräg darüber und darunter. Platz 4 hat also
sechs Nachbarn, Platz 0 nur zwei. Die Liste `NACHBARN` ist fertig.

Ein Kunde wünscht sich eine Zahl `farbanzahl` von **verschiedenen** Farben im Beet und vergibt für einige Farbpaare 1 bis 3
**Bonuspunkte**. Jedes Paar benachbarter Blumen bringt die Punkte seines Farbpaars (0, wenn es keinen Wunsch dazu gibt).
Gesucht ist ein Beet mit genau `farbanzahl` Farben und der größten Punktsumme.

**Vereinfachung:** Wir pflanzen nur die Farben, die in den Wünschen vorkommen, und der Kunde wünscht sich höchstens so viele
Farben. In der echten Aufgabe kann er mehr Farben verlangen. Dann braucht man „Füllfarben“, die keine Punkte bringen und
untereinander austauschbar sind: wieder eine Symmetrie. Das Lösungsnotebook zeigt, wie man das macht.

Die Dateien enthalten in der ersten Zeile `farbanzahl`, in der zweiten die Zahl der Wünsche und danach je Wunsch zwei
Farben und die Punkte. `blumen1.txt` und `blumen2.txt` sind Beispiele der echten Aufgabe, `blumen_eigen.txt` hat vier Farben.

Wir pflanzen Platz für Platz, `beet` ist die Liste der schon gepflanzten Farben. Kommt Platz `i` dazu, zählen die Paare mit
seinen **früheren** Nachbarn (`FRUEHERE[i]`). So wird jedes Paar genau einmal gezählt.

#### Aufgabe E.1

Schreibe `pflanze(beet, punkte)`, die alle erlaubten Beete durchsucht und das beste in `beste` merkt. Brich ab, wenn das Beet
schon zu viele verschiedene Farben hat oder wenn die freien Plätze nicht mehr reichen, um auf `farbanzahl` Farben zu kommen.

**Erwartete Ausgabe:**
```
daten/blumen1.txt : 36 Punkte, 4522 Aufrufe, ['rot', 'tuerkis', 'tuerkis', 'rot', 'rot', 'rot', 'tuerkis', 'tuerkis', 'rot']
daten/blumen2.txt : 32 Punkte, 29515 Aufrufe, ['gruen', 'rot', 'rot', 'tuerkis', 'tuerkis', 'tuerkis', 'rot', 'rot', 'tuerkis']
daten/blumen_eigen.txt : 37 Punkte, 343397 Aufrufe, ['blau', 'gelb', 'rot', 'gruen', 'blau', 'blau', 'gelb', 'rot', 'blau']
```

In [ ]:
NACHBARN = [[1, 2], [0, 2, 3, 4], [0, 1, 4, 5], [1, 4, 6], [1, 2, 3, 5, 6, 7], [2, 4, 7], [3, 4, 7, 8],
            [4, 5, 6, 8], [6, 7]]
FRUEHERE = []
for i in range(9):
    frueher = []
    for j in NACHBARN[i]:
        if j < i:
            frueher.append(j)
    FRUEHERE.append(frueher)

def lies_wuensche(dateiname):
    teile = open(dateiname).read().split()
    farbanzahl = int(teile[0])
    bonus = {}
    for k in range(int(teile[1])):
        farbe1 = teile[2 + 3 * k]
        farbe2 = teile[3 + 3 * k]
        punkte = int(teile[4 + 3 * k])
        bonus[(farbe1, farbe2)] = punkte          # beide Reihenfolgen eintragen
        bonus[(farbe2, farbe1)] = punkte
    return farbanzahl, bonus

def zeige_beet(beet):
    for reihe in [[0], [1, 2], [3, 4, 5], [6, 7], [8]]:
        text = ' ' * 4 * (3 - len(reihe))
        for platz in reihe:
            text = text + beet[platz].ljust(8)
        print(text.rstrip())

def pflanze(beet, punkte):
    aufrufe[0] = aufrufe[0] + 1
    # dein Code

DATEIEN = ['daten/blumen1.txt', 'daten/blumen2.txt', 'daten/blumen_eigen.txt']
for datei in DATEIEN:
    farbanzahl, bonus = lies_wuensche(datei)
    farben = []
    for farbe1, farbe2 in bonus:
        if farbe1 not in farben:
            farben.append(farbe1)
    farben.sort()
    beste = [-1, []]
    aufrufe[0] = 0
    pflanze([], 0)
    print(datei, ':', beste[0], 'Punkte,', aufrufe[0], 'Aufrufe,', beste[1])

<details>
<summary>Tipp anzeigen</summary>

`i = len(beet)` ist der nächste Platz, `len(set(beet))` die Zahl der verschiedenen Farben. Nach den beiden Abbrüchen: Ist
`i == 9`, mit `beste` vergleichen. Sonst für jede Farbe die neuen Punkte `bonus.get((farbe, beet[j]), 0)` für alle `j` in
`FRUEHERE[i]` addieren, setzen, weitersuchen, zurücknehmen. `bonus.get(paar, 0)` liefert 0, wenn das Paar nicht im
Wörterbuch steht.

</details>

<details>
<summary>Lösung anzeigen (erst selbst probieren!)</summary>

```python
NACHBARN = [[1, 2], [0, 2, 3, 4], [0, 1, 4, 5], [1, 4, 6], [1, 2, 3, 5, 6, 7], [2, 4, 7], [3, 4, 7, 8],
            [4, 5, 6, 8], [6, 7]]
FRUEHERE = []
for i in range(9):
    frueher = []
    for j in NACHBARN[i]:
        if j < i:
            frueher.append(j)
    FRUEHERE.append(frueher)

def lies_wuensche(dateiname):
    teile = open(dateiname).read().split()
    farbanzahl = int(teile[0])
    bonus = {}
    for k in range(int(teile[1])):
        farbe1 = teile[2 + 3 * k]
        farbe2 = teile[3 + 3 * k]
        punkte = int(teile[4 + 3 * k])
        bonus[(farbe1, farbe2)] = punkte          # beide Reihenfolgen eintragen
        bonus[(farbe2, farbe1)] = punkte
    return farbanzahl, bonus

def zeige_beet(beet):
    for reihe in [[0], [1, 2], [3, 4, 5], [6, 7], [8]]:
        text = ' ' * 4 * (3 - len(reihe))
        for platz in reihe:
            text = text + beet[platz].ljust(8)
        print(text.rstrip())

def pflanze(beet, punkte):
    aufrufe[0] = aufrufe[0] + 1
    i = len(beet)
    verschieden = len(set(beet))
    if verschieden > farbanzahl or verschieden + (9 - i) < farbanzahl:
        return                                    # Farbanzahl nicht mehr zu schaffen
    if i == 9:
        if punkte > beste[0]:
            beste[0] = punkte
            beste[1] = list(beet)
        return
    for farbe in farben:
        dazu = 0
        for j in FRUEHERE[i]:                     # neue Paare mit früheren Nachbarn
            dazu = dazu + bonus.get((farbe, beet[j]), 0)
        beet.append(farbe)
        pflanze(beet, punkte + dazu)
        beet.pop()

DATEIEN = ['daten/blumen1.txt', 'daten/blumen2.txt', 'daten/blumen_eigen.txt']
for datei in DATEIEN:
    farbanzahl, bonus = lies_wuensche(datei)
    farben = []
    for farbe1, farbe2 in bonus:
        if farbe1 not in farben:
            farben.append(farbe1)
    farben.sort()
    beste = [-1, []]
    aufrufe[0] = 0
    pflanze([], 0)
    print(datei, ':', beste[0], 'Punkte,', aufrufe[0], 'Aufrufe,', beste[1])
```

Die 36 und 32 Punkte stimmen mit den Lösungshinweisen überein. Bei `blumen_eigen.txt` sind es schon über 300 000 Aufrufe,
denn mit 4 Farben gibt es 4⁹ = 262 144 volle Beete.

</details>

**Eine Schranke für das Beet.** Das Beet hat 16 Nachbarpaare. Ist Platz `i` als nächster dran, sind die Paare mit späteren
Plätzen noch offen. Ihre Zahl steht in `OFFEN[i]` (zum Beispiel `OFFEN[0] = 16`, `OFFEN[9] = 0`). Jedes offene Paar bringt
höchstens so viele Punkte wie der größte Bonus `maximum`. Also kann das Beet höchstens `punkte + OFFEN[i] * maximum` Punkte
erreichen.

#### Aufgabe E.2

Schreibe `pflanze2` wie `pflanze`, aber mit der Schranke: Ist `punkte + OFFEN[i] * maximum <= beste[0]`, brich ab. Die Schranke
gehört nach den beiden Abbrüchen für die Farbanzahl und nach dem Fall `i == 9`.

**Erwartete Ausgabe:**
```
OFFEN: [16, 16, 15, 13, 12, 9, 7, 5, 2, 0]
daten/blumen1.txt : 36 Punkte, 2137 Aufrufe
daten/blumen2.txt : 32 Punkte, 2737 Aufrufe
daten/blumen_eigen.txt : 37 Punkte, 10261 Aufrufe
```

In [ ]:
OFFEN = []
for i in range(10):
    summe = 0
    for platz in range(i, 9):
        summe = summe + len(FRUEHERE[platz])
    OFFEN.append(summe)
print('OFFEN:', OFFEN)

def pflanze2(beet, punkte):
    aufrufe[0] = aufrufe[0] + 1
    # dein Code

for datei in DATEIEN:
    farbanzahl, bonus = lies_wuensche(datei)
    farben = []
    for farbe1, farbe2 in bonus:
        if farbe1 not in farben:
            farben.append(farbe1)
    farben.sort()
    maximum = max(bonus.values())
    beste = [-1, []]
    aufrufe[0] = 0
    pflanze2([], 0)
    print(datei, ':', beste[0], 'Punkte,', aufrufe[0], 'Aufrufe')

<details>
<summary>Lösung anzeigen (erst selbst probieren!)</summary>

```python
OFFEN = []
for i in range(10):
    summe = 0
    for platz in range(i, 9):
        summe = summe + len(FRUEHERE[platz])
    OFFEN.append(summe)
print('OFFEN:', OFFEN)

def pflanze2(beet, punkte):
    aufrufe[0] = aufrufe[0] + 1
    i = len(beet)
    verschieden = len(set(beet))
    if verschieden > farbanzahl or verschieden + (9 - i) < farbanzahl:
        return
    if i == 9:
        if punkte > beste[0]:
            beste[0] = punkte
            beste[1] = list(beet)
        return
    if punkte + OFFEN[i] * maximum <= beste[0]:   # Schranke: selbst im besten Fall nicht besser
        return
    for farbe in farben:
        dazu = 0
        for j in FRUEHERE[i]:
            dazu = dazu + bonus.get((farbe, beet[j]), 0)
        beet.append(farbe)
        pflanze2(beet, punkte + dazu)
        beet.pop()

for datei in DATEIEN:
    farbanzahl, bonus = lies_wuensche(datei)
    farben = []
    for farbe1, farbe2 in bonus:
        if farbe1 not in farben:
            farben.append(farbe1)
    farben.sort()
    maximum = max(bonus.values())
    beste = [-1, []]
    aufrufe[0] = 0
    pflanze2([], 0)
    print(datei, ':', beste[0], 'Punkte,', aufrufe[0], 'Aufrufe')
```

</details>

**Spiegeln.** Spiegelt man das Beet an der senkrechten Mittellinie, tauschen die Plätze 1 und 2, 3 und 5, 6 und 7. Die
Nachbarschaften bleiben dieselben, also auch die Punkte und die Farbanzahl. Von einem Beet und seinem Spiegelbild hat
eines auf Platz 1 eine Farbe, die im Alphabet nicht nach der auf Platz 2 kommt. Es genügt also, nur Beete mit
`beet[1] <= beet[2]` zu suchen. Texte vergleicht Python wie im Wörterbuch: `'blau' < 'rot'` ist `True`.

#### Aufgabe E.3

Schreibe `pflanze3` wie `pflanze2`, aber überspringe auf Platz 2 jede Farbe, die im Alphabet vor `beet[1]` kommt. Zeige das
beste Beet für `blumen_eigen.txt` mit `zeige_beet`.

**Erwartete Ausgabe:**
```
daten/blumen1.txt : 36 Punkte, 1591 Aufrufe
daten/blumen2.txt : 32 Punkte, 1966 Aufrufe
daten/blumen_eigen.txt : 37 Punkte, 7177 Aufrufe
        blau
    gelb    rot
gruen   blau    blau
    gelb    rot
        blau
```

In [ ]:
def pflanze3(beet, punkte):
    aufrufe[0] = aufrufe[0] + 1
    # dein Code

for datei in DATEIEN:
    farbanzahl, bonus = lies_wuensche(datei)
    farben = []
    for farbe1, farbe2 in bonus:
        if farbe1 not in farben:
            farben.append(farbe1)
    farben.sort()
    maximum = max(bonus.values())
    beste = [-1, []]
    aufrufe[0] = 0
    pflanze3([], 0)
    print(datei, ':', beste[0], 'Punkte,', aufrufe[0], 'Aufrufe')
if len(beste[1]) == 9:
    zeige_beet(beste[1])

<details>
<summary>Tipp anzeigen</summary>

In der Schleife über die Farben als Erstes: `if i == 2 and farbe < beet[1]: continue`. `continue` springt zur nächsten Farbe.

</details>

<details>
<summary>Lösung anzeigen (erst selbst probieren!)</summary>

```python
def pflanze3(beet, punkte):
    aufrufe[0] = aufrufe[0] + 1
    i = len(beet)
    verschieden = len(set(beet))
    if verschieden > farbanzahl or verschieden + (9 - i) < farbanzahl:
        return
    if i == 9:
        if punkte > beste[0]:
            beste[0] = punkte
            beste[1] = list(beet)
        return
    if punkte + OFFEN[i] * maximum <= beste[0]:
        return
    for farbe in farben:
        if i == 2 and farbe < beet[1]:            # Spiegelbild wird schon anders gesucht
            continue
        dazu = 0
        for j in FRUEHERE[i]:
            dazu = dazu + bonus.get((farbe, beet[j]), 0)
        beet.append(farbe)
        pflanze3(beet, punkte + dazu)
        beet.pop()

for datei in DATEIEN:
    farbanzahl, bonus = lies_wuensche(datei)
    farben = []
    for farbe1, farbe2 in bonus:
        if farbe1 not in farben:
            farben.append(farbe1)
    farben.sort()
    maximum = max(bonus.values())
    beste = [-1, []]
    aufrufe[0] = 0
    pflanze3([], 0)
    print(datei, ':', beste[0], 'Punkte,', aufrufe[0], 'Aufrufe')
zeige_beet(beste[1])
```

</details>

#### Frage E.4

Für die Dokumentation:

1. Warum verliert die Schranke `punkte + OFFEN[i] * maximum` keine bessere Lösung?
2. Warum verliert die Bedingung `beet[1] <= beet[2]` keine Punktzahl, obwohl sie Beete weglässt?
3. Die Schranke ist recht grob. Wie könnte man sie verbessern?

✏️ *Deine Antwort (Doppelklick zum Bearbeiten):*

<details>
<summary>Lösung anzeigen (erst selbst überlegen!)</summary>

1. Die Punkte der schon gepflanzten Plätze stehen fest. Jedes der `OFFEN[i]` noch offenen Paare bringt höchstens `maximum`
   Punkte. Mehr als `punkte + OFFEN[i] * maximum` geht also nicht, und wenn das nicht mehr als `beste[0]` ist, kann keine
   Fortsetzung besser werden.
2. Zu jedem weggelassenen Beet wird sein Spiegelbild gesucht. Es hat dieselben Nachbarpaare, also dieselben Punkte und
   dieselbe Farbanzahl, und es erfüllt die Bedingung. Die beste Punktzahl wird also noch gefunden, nur vielleicht als
   gespiegeltes Beet.
3. Zum Beispiel: Bei jedem offenen Paar, das einen schon gepflanzten Platz berührt, kennt man eine Farbe schon und kann
   den größten Bonus **dieser** Farbe nehmen statt des größten überhaupt. Oder man nutzt, dass Paare ohne Wunsch nichts
   bringen. Jede Verbesserung muss wieder optimistisch bleiben, und man prüft sie, indem man die Ergebnisse mit `pflanze`
   vergleicht.

</details>

## Wettbewerbs-Ecke 2: Rominos (für Schnelle)

*Angelehnt an Aufgabe 5 „Rominos“ der 1. Runde des 38. Bundeswettbewerbs Informatik (2019/20).*

Ruth erfindet neue Figuren aus Quadraten, die **Rominos**. Die Quadrate dürfen nicht nur an Seiten, sondern auch an
**Ecken** aneinanderhängen. Ein Romino muss mindestens ein Paar von Quadraten haben, die sich **nur an einer Ecke**
berühren, und zwar so, dass sie nicht beide an den Seiten **desselben dritten** Quadrats der Figur liegen. Figuren, die
durch Verschieben, Drehen oder Spiegeln zur Deckung kommen, gelten als gleich. Laut Aufgabe gibt es genau drei Rominos aus
3 Quadraten. Wie viele sind es bei 4 und 5 Quadraten?

Für das Paar `(x, y)` und `(x + 1, y + 1)` sind die beiden „dritten“ Quadrate, die an beiden mit einer Seite anliegen,
`(x + 1, y)` und `(x, y + 1)`. Für `(x, y)` und `(x + 1, y - 1)` sind es `(x + 1, y)` und `(x, y - 1)`.

Alles andere hast du schon: `wachse` mit **acht** Richtungen (Seiten und Ecken) erzeugt alle Figuren, deren Quadrate an
Seiten oder Ecken zusammenhängen. Davon behältst du die Rominos und zählst sie mit `kanonisch`.

#### Aufgabe E.5

Schreibe `ist_romino(figur)`. Sie prüft für jedes Feld `(x, y)` die beiden Diagonalen nach rechts oben (`dy = 1`) und
rechts unten (`dy = -1`).

**Erwartete Ausgabe:**
```
True
False
True
False
```

In [ ]:
def ist_romino(figur):
    # dein Code
    return False

print(ist_romino([(0, 0), (1, 1)]))                    # zwei Quadrate an einer Ecke
print(ist_romino([(0, 0), (1, 1), (1, 0)]))            # das dritte Quadrat liegt an beiden an
print(ist_romino([(0, 0), (1, 0), (2, 0), (3, 1)]))    # Stab mit einem Quadrat an der Ecke
print(ist_romino([(0, 0), (1, 0), (2, 0), (3, 0)]))    # nur Seiten

<details>
<summary>Tipp anzeigen</summary>

`for x, y in figur:` und `for dy in [1, -1]:`. Ist `(x + 1, y + dy)` in der Figur und sind `(x + 1, y)` und `(x, y + dy)` beide
nicht in der Figur, gib `True` zurück.

</details>

<details>
<summary>Lösung anzeigen (erst selbst probieren!)</summary>

```python
def ist_romino(figur):
    for x, y in figur:
        for dy in [1, -1]:
            if (x + 1, y + dy) in figur:                                  # Nachbar an der Ecke
                if (x + 1, y) not in figur and (x, y + dy) not in figur:  # kein drittes Quadrat dazwischen
                    return True
    return False

print(ist_romino([(0, 0), (1, 1)]))                    # zwei Quadrate an einer Ecke
print(ist_romino([(0, 0), (1, 1), (1, 0)]))            # das dritte Quadrat liegt an beiden an
print(ist_romino([(0, 0), (1, 0), (2, 0), (3, 1)]))    # Stab mit einem Quadrat an der Ecke
print(ist_romino([(0, 0), (1, 0), (2, 0), (3, 0)]))    # nur Seiten
```

Warum nur nach rechts? Jedes Eckenpaar hat ein linkes und ein rechtes Quadrat. Vom linken aus wird es gefunden, also
genügt eine Richtung.

</details>

#### Aufgabe E.6

Zähle die Rominos aus 2 bis 6 Quadraten und zeichne die Rominos aus 4 Quadraten.

**Erwartete Ausgabe:**
```
2 Quadrate: 4 Lagen geprüft, 1 Rominos
3 Quadrate: 20 Lagen geprüft, 3 Rominos
4 Quadrate: 110 Lagen geprüft, 17 Rominos
5 Quadrate: 638 Lagen geprüft, 82 Rominos
6 Quadrate: 3832 Lagen geprüft, 489 Rominos
.#   #.              .#               ..#   .#
#.   .#   .#   ..#   .#   .#.   .##   .#.   #.
#.   #.   #.   ##.   #.   #.#   #..   #..   .#
#.   #.   ##   #..   #.   #..   #..   #..   #.

            #.         ..#   .#.   ...#
#..   #..   .#   .#.   .#.   ..#   ..#.   .#.
.#.   .##   .#   .##   .#.   .#.   .#..   #.#
#.#   #..   #.   #..   #..   #..   #...   .#.
```

In [ ]:
ACHT = [(1, 0), (-1, 0), (0, 1), (0, -1), (1, 1), (1, -1), (-1, 1), (-1, -1)]

# dein Code

<details>
<summary>Tipp anzeigen</summary>

Wie in 3.4, aber mit `ACHT` statt `SEITEN`, und in die Menge kommt `kanonisch(figur)` nur, wenn `ist_romino(figur)` stimmt.

</details>

<details>
<summary>Lösung anzeigen (erst selbst probieren!)</summary>

```python
ACHT = [(1, 0), (-1, 0), (0, 1), (0, -1), (1, 1), (1, -1), (-1, 1), (-1, -1)]

figuren_acht = {((0, 0),)}
for n in range(2, 7):
    figuren_acht = wachse(figuren_acht, ACHT)
    rominos = set()
    for figur in figuren_acht:
        if ist_romino(figur):
            rominos.add(kanonisch(figur))
    print(n, 'Quadrate:', len(figuren_acht), 'Lagen geprüft,', len(rominos), 'Rominos')
    if n == 4:
        vier = sorted(rominos)
zeichne(vier[:9])
print()
zeichne(vier[9:])
```

Für 3 Quadrate stimmt das mit der Aufgabe überein. Die Lösungshinweise nennen 17 Rominos aus 4 Quadraten, 82 aus 5 und
489 aus 6, genau wie unser Programm. Die Zahl der Lagen wächst mit jedem Quadrat etwa um den Faktor 6. Für 10 Quadrate
wären es über sechs Millionen, dann wird es im Browser zu langsam. Das Lösungsnotebook zeigt einen Weg, der jede Lage nur
einmal erzeugt.

</details>

## Das hast du gelernt

- **Beste Lösung merken:** `beste = [wert, loesung]`, bei jeder besseren Lösung eine Kopie speichern.
- **Schranke (Branch and Bound):** Wie gut kann ein angefangener Versuch **höchstens** noch werden? Ist das nicht besser als
  das Beste bisher, abbrechen. Die Schranke muss **optimistisch** sein (nie zu klein), sonst wird das Ergebnis falsch.
- **Reihenfolge:** Gute Lösungen früh finden, dann schneidet die Schranke mehr ab. `sorted(liste, key=lambda d: d[2],
  reverse=True)` sortiert nach einem Teil der Tupel.
- **Symmetrie brechen:** nur einen Vertreter suchen (linke Hälfte, `beet[1] <= beet[2]`) und sauber begründen, dass nichts
  verloren geht.
- **Normalform:** verschieben, sortieren, als Tupel speichern, von allen Lagen die kleinste nehmen. Dann zählt eine Menge
  jede Figur einmal.
- **Prüfen:** Ergebnisse mit und ohne Abkürzung vergleichen und mit bekannten Zahlen (19 Lagen, 5 Tetrominos, 12 Pentominos).

### Aufgaben der 1. Runde dazu

Bearbeite eine Aufgabe zuerst selbst und sieh dir das Lösungsnotebook erst danach an.

- [Blumenbeet](https://ktheu.github.io/bwinfN/notebooks/index.html?path=bwinf1/38_Blumenbeet/blumenbeet_loesung.ipynb) (38. BwInf): mit Füllfarben und allen Beispielen
- [Rominos](https://ktheu.github.io/bwinfN/notebooks/index.html?path=bwinf1/38_Rominos/rominos_loesung.ipynb) (38.): jede Lage nur einmal erzeugen, Anzahlen bis 10
- [Dreieckspuzzle](https://ktheu.github.io/bwinfN/notebooks/index.html?path=bwinf1/39_Dreieckspuzzle/dreieckspuzzle_loesung.ipynb) (39.): jede Lösung dreimal
- [Telepaartie](https://ktheu.github.io/bwinfN/notebooks/index.html?path=bwinf1/38_Telepaartie/telepaartie_loesung.ipynb) (38.): Zustände in Normalform
- [Twist](https://ktheu.github.io/bwinfN/notebooks/index.html?path=bwinf1/37_Twist/twist_loesung.ipynb) (37.)
- [Geldtransporter](https://ktheu.github.io/bwinfN/notebooks/index.html?path=bwinf1/31_Geldtransporter/geldtransporter_loesung.ipynb) (31.), [Schrebergärten](https://ktheu.github.io/bwinfN/notebooks/index.html?path=bwinf1/37_Schrebergaerten/schrebergaerten_loesung.ipynb) (37.) und
  [Großstadtbauern](https://ktheu.github.io/bwinfN/notebooks/index.html?path=bwinf1/44_Grossstadtbauern/grossstadtbauern_loesung.ipynb) (44.): Schranken

Das nächste Notebook zeigt, wie man bei Rätseln wie Sudoku **logisch schließt**: erst alles Zwingende eintragen und
erst dann raten.